# Ensemble: Combining Models A, B, C, D

This notebook loads all 4 individually trained models and evaluates
multiple ensemble strategies:

1. **Hard Voting** -- majority vote on predicted labels
2. **Soft Voting** -- average predicted probabilities
3. **Weighted Voting** -- weights proportional to individual CV scores
4. **Stacking** -- meta-learner trained on base model predictions

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import VotingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression

from src.data_loader import load_dataset, get_train_test_split
from src.evaluation import (
    evaluate_model,
    plot_full_evaluation,
    compare_models,
)
from src.model_registry import load_all_models, load_model_metadata

sns.set_theme(style="whitegrid")
%matplotlib inline

## 1. Load Data & Models

In [ ]:
df = load_dataset()
X_train, X_test, y_train, y_test = get_train_test_split(df)

In [ ]:
models = load_all_models()
print(f"Models loaded: {list(models.keys())}")

## 2. Individual Model Performance

Evaluate each model on the test set for baseline comparison.

In [ ]:
individual_metrics = []
for name, model in models.items():
    m = plot_full_evaluation(model, X_test, y_test, model_name=name)
    individual_metrics.append(m)

df_individual = compare_models(individual_metrics)
df_individual

## 3. Hard Voting Ensemble

In [ ]:
estimators = [(name, model) for name, model in models.items()]

hard_voting = VotingClassifier(
    estimators=estimators,
    voting="hard",
)

# VotingClassifier needs to be fit, but since the estimators are already fitted,
# we can use the prefit trick:
# Option 1: Refit on training data (recommended for consistency)
# hard_voting.fit(X_train, y_train)
#
# Option 2: Manually aggregate predictions (if refitting is not desired)
# y_preds = np.array([m.predict(X_test) for m in models.values()])
# y_hard = np.apply_along_axis(lambda x: np.bincount(x).argmax(), axis=0, arr=y_preds)

# TODO: Uncomment your preferred approach
# hard_metrics = plot_full_evaluation(hard_voting, X_test, y_test, model_name="Hard Voting")

## 4. Soft Voting Ensemble

In [ ]:
soft_voting = VotingClassifier(
    estimators=estimators,
    voting="soft",
)

# soft_voting.fit(X_train, y_train)
# soft_metrics = plot_full_evaluation(soft_voting, X_test, y_test, model_name="Soft Voting")

## 5. Weighted Voting Ensemble

Weights derived from each model's cross-validation score.

In [ ]:
# Load CV scores from metadata
weights = []
for name in models.keys():
    meta = load_model_metadata(name)
    cv = meta.get("cv_score", 0.5)
    weights.append(cv)
    print(f"{name}: CV score = {cv:.4f}")

print(f"Weights: {weights}")

In [ ]:
weighted_voting = VotingClassifier(
    estimators=estimators,
    voting="soft",
    weights=weights,
)

# weighted_voting.fit(X_train, y_train)
# weighted_metrics = plot_full_evaluation(weighted_voting, X_test, y_test, model_name="Weighted Voting")

## 6. Stacking Ensemble

Uses a Logistic Regression meta-learner on top of the base model predictions.

In [ ]:
stacking = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(random_state=13),
    cv=5,
    passthrough=False,
)

# stacking.fit(X_train, y_train)
# stacking_metrics = plot_full_evaluation(stacking, X_test, y_test, model_name="Stacking")

## 7. Comparison: All Models & Ensembles

In [ ]:
# TODO: Collect all metrics into a single comparison table
# all_metrics = individual_metrics + [hard_metrics, soft_metrics, weighted_metrics, stacking_metrics]
# df_comparison = compare_models(all_metrics)
# df_comparison

In [ ]:
# TODO: Bar plot of F1 scores
# fig, ax = plt.subplots(figsize=(10, 6))
# df_comparison.plot.barh(x="model", y="f1", ax=ax, color="steelblue")
# ax.set_xlabel("F1 Score")
# ax.set_title("Model Comparison: F1 Score")
# plt.tight_layout()
# plt.show()

## 8. Conclusions

- TODO: Best ensemble strategy and why
- TODO: Which individual models contributed most
- TODO: Final recommendation